# YouTube Metadata Collection

**Collect engagement metrics for songs with audio features**

**Metadata to collect:**
1. view_count
2. like_count
3. comment_count
4. channel_follower_count
5. duration
6. upload_date
7. heatmap (engagement pattern)
8. channel_is_verified
9. tags
10. categories

**Method:** yt-dlp --dump-json (no downloads, no API quota)

**Estimated time:** 5-6 hours for ~44,000 videos with 8 workers

## 1. Setup & Imports

In [1]:
import subprocess
import json
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import time

print("✓ Libraries loaded")

✓ Libraries loaded


## 2. Configuration

In [2]:
# Configuration
NUM_WORKERS = 8  # Conservative for metadata collection
BATCH_SIZE = 500  # Save progress every 500 videos

# Paths
FEATURES_CSV = Path("../data/raw/audio_features.csv")
DOWNLOAD_CSV = Path("../data/raw/combined_match_download_results.csv")
OUTPUT_CSV = Path("../data/raw/youtube_metadata.csv")

print(f"Workers: {NUM_WORKERS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Output: {OUTPUT_CSV}")

Workers: 8
Batch size: 500
Output: ..\data\raw\youtube_metadata.csv


## 3. Load Songs with Features

In [3]:
# Load songs that have audio features
features_df = pd.read_csv(FEATURES_CSV)
download_df = pd.read_csv(DOWNLOAD_CSV)

print(f"Songs with features: {len(features_df):,}")
print(f"Download results: {len(download_df):,}")

# Merge to get YouTube video IDs
songs_to_process = features_df.merge(
    download_df[['track_id', 'youtube_video_id', 'youtube_title']], 
    on='track_id', 
    how='inner'
)

print(f"\nSongs to collect metadata for: {len(songs_to_process):,}")
print(f"\nSample:")
print(songs_to_process[['track_id', 'youtube_video_id', 'youtube_title']].head())

Songs with features: 44,271
Download results: 47,344

Songs to collect metadata for: 45,796

Sample:
                 track_id youtube_video_id  \
0  005U5ZHLoA4OEkWewAa32S      _sPGLRnwf8s   
1  000CC8EParg64OmTxVnZ0p      xAuWOuUy6eQ   
2  0068lzo1xXa9ED8ThypHU1      ZeM8O_nz6a0   
3  004iWPkSRbvOEvAPLWHl9M      hsvQEsY_a9A   
4  006rHBBNLJMpQs8fRC2GDe      AkjqLJAs4Ds   

                                       youtube_title  
0                        Strength In Your Conviction  
1                 Glee Cast - Sing! (Official Audio)  
2        Horizon Blue & Carston - Got You On My Mind  
3                                        Mister Love  
4  Calcinha Preta feat. @gusttavolimaoficial - Ag...  


## 4. Check for Existing Progress

In [4]:
# Check if we have existing metadata
if OUTPUT_CSV.exists():
    existing_metadata = pd.read_csv(OUTPUT_CSV)
    already_processed = set(existing_metadata['youtube_video_id'].values)
    
    print(f"Found existing metadata file")
    print(f"Already processed: {len(already_processed):,} videos")
    
    # Filter to unprocessed
    pending = songs_to_process[
        ~songs_to_process['youtube_video_id'].isin(already_processed)
    ]
    
    print(f"Remaining to process: {len(pending):,} videos")
else:
    print("No existing metadata file - starting fresh")
    pending = songs_to_process
    existing_metadata = None

print(f"\nTotal videos to process: {len(pending):,}")

Found existing metadata file
Already processed: 26,997 videos
Remaining to process: 11,689 videos

Total videos to process: 11,689


## 5. Metadata Collection Function

In [5]:
def collect_metadata(video_id, track_id):
    """
    Collect YouTube metadata for a video using yt-dlp --dump-json
    No download, no API quota usage
    """
    try:
        # Build yt-dlp command
        cmd = [
            'yt-dlp',
            f'https://www.youtube.com/watch?v={video_id}',
            '--dump-json',
            '--skip-download',
            '--no-warnings'
        ]
        
        # Run command with timeout
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        # Check for blocking
        if 'Sign in to confirm' in result.stderr or 'bot' in result.stderr.lower():
            return {
                'track_id': track_id,
                'youtube_video_id': video_id,
                'collection_success': False,
                'error_message': 'YouTube blocked - bot detection'
            }
        
        # Parse JSON output
        if result.stdout:
            data = json.loads(result.stdout)
            
            # Extract metadata
            metadata = {
                'track_id': track_id,
                'youtube_video_id': video_id,
                
                # Engagement metrics
                'view_count': data.get('view_count'),
                'like_count': data.get('like_count'),
                'comment_count': data.get('comment_count'),
                
                # Channel info
                'channel_follower_count': data.get('channel_follower_count'),
                'channel_is_verified': data.get('channel_is_verified'),
                'channel_id': data.get('channel_id'),
                'uploader': data.get('uploader'),
                
                # Video info
                'duration': data.get('duration'),
                'upload_date': data.get('upload_date'),
                'title': data.get('title'),
                
                # Categories and tags
                'categories': json.dumps(data.get('categories', [])),
                'tags': json.dumps(data.get('tags', [])),
                
                # Heatmap (engagement pattern)
                'heatmap': json.dumps(data.get('heatmap')),
                
                # Age restriction
                'age_limit': data.get('age_limit'),
                
                'collection_success': True,
                'error_message': ''
            }
            
            return metadata
        else:
            return {
                'track_id': track_id,
                'youtube_video_id': video_id,
                'collection_success': False,
                'error_message': 'No JSON output from yt-dlp'
            }
            
    except subprocess.TimeoutExpired:
        return {
            'track_id': track_id,
            'youtube_video_id': video_id,
            'collection_success': False,
            'error_message': 'Timeout after 30 seconds'
        }
    except json.JSONDecodeError as e:
        return {
            'track_id': track_id,
            'youtube_video_id': video_id,
            'collection_success': False,
            'error_message': f'JSON parse error: {str(e)}'
        }
    except Exception as e:
        return {
            'track_id': track_id,
            'youtube_video_id': video_id,
            'collection_success': False,
            'error_message': str(e)
        }

print("✓ Metadata collection function defined")

✓ Metadata collection function defined


## 6. Test on Single Video

In [6]:
# Test on first video
if len(pending) > 0:
    test_row = pending.iloc[0]
    test_video_id = test_row['youtube_video_id']
    test_track_id = test_row['track_id']
    
    print(f"Testing metadata collection on video: {test_video_id}")
    print(f"Track: {test_row['youtube_title']}\n")
    
    start_time = time.time()
    test_metadata = collect_metadata(test_video_id, test_track_id)
    elapsed = time.time() - start_time
    
    if test_metadata.get('collection_success'):
        print(f"✓ Collection successful! ({elapsed:.2f} seconds)\n")
        print("Sample metadata:")
        
        # Helper function to format numbers or show N/A
        def fmt(val):
            if val is None:
                return 'N/A'
            elif isinstance(val, (int, float)):
                return f'{val:,}'
            else:
                return str(val)
        
        print(f"  Views: {fmt(test_metadata.get('view_count'))}")
        print(f"  Likes: {fmt(test_metadata.get('like_count'))}")
        print(f"  Comments: {fmt(test_metadata.get('comment_count'))}")
        print(f"  Channel followers: {fmt(test_metadata.get('channel_follower_count'))}")
        print(f"  Duration: {fmt(test_metadata.get('duration'))} seconds")
        print(f"  Upload date: {test_metadata.get('upload_date', 'N/A')}")
        print(f"  Channel verified: {test_metadata.get('channel_is_verified', 'N/A')}")
        
        # Show all collected fields
        print(f"\nAll collected fields:")
        for key, value in test_metadata.items():
            if key not in ['track_id', 'youtube_video_id', 'collection_success', 'error_message']:
                if value is not None and value != '' and value != '[]' and value != 'null':
                    print(f"  {key}: {value}")
        
        # Projection
        total_time_hours = (len(pending) * elapsed) / (NUM_WORKERS * 3600)
        print(f"\nProjected time for {len(pending):,} videos with {NUM_WORKERS} workers:")
        print(f"  {total_time_hours:.1f} hours ({total_time_hours/24:.1f} days)")
    else:
        print(f"✗ Collection failed!")
        print(f"  Error: {test_metadata.get('error_message', 'Unknown')}")
        print(f"\n⚠️ If blocked, try:")
        print(f"  1. Wait 3-5 days for block to expire")
        print(f"  2. Use mobile hotspot")
        print(f"  3. Use VPN")
else:
    print("No videos to process!")

Testing metadata collection on video: JpFrbKWv-O0
Track: Guy Clark - Soldier's Joy (Official Audio)

✓ Collection successful! (3.83 seconds)

Sample metadata:
  Views: 16,200
  Likes: 186
  Comments: 9
  Channel followers: 14,500
  Duration: 220 seconds
  Upload date: 20230420
  Channel verified: True

All collected fields:
  view_count: 16200
  like_count: 186
  comment_count: 9
  channel_follower_count: 14500
  channel_is_verified: True
  channel_id: UCcUS13mmMOw_o4YySN8zRcA
  uploader: Guy Clark
  duration: 220
  upload_date: 20230420
  title: Guy Clark - Soldier's Joy (Official Audio)
  categories: ["People & Blogs"]
  tags: ["guy clark", "old friends", "country", "folk", "singer-songwriter", "americana"]
  age_limit: 0

Projected time for 11,689 videos with 8 workers:
  1.6 hours (0.1 days)


## 7. Parallel Metadata Collection

**⚠️ Uncomment and run only after test succeeds!**

In [7]:
# Parallel metadata collection
def process_batch_metadata(videos_df, num_workers=8, batch_size=500):
    """
    Process videos in batches with parallel workers
    """
    all_metadata = []
    
    # Load existing if available
    if OUTPUT_CSV.exists():
        existing = pd.read_csv(OUTPUT_CSV)
        all_metadata = existing.to_dict('records')
        print(f"Loaded {len(all_metadata):,} existing metadata records\n")
    
    total_videos = len(videos_df)
    
    # Process in batches
    for batch_start in range(0, total_videos, batch_size):
        batch_end = min(batch_start + batch_size, total_videos)
        batch_videos = videos_df.iloc[batch_start:batch_end]
        
        print(f"\n{'='*70}")
        print(f"Batch: {batch_start + 1}-{batch_end} of {total_videos:,}")
        print(f"Workers: {num_workers}")
        print(f"{'='*70}\n")
        
        batch_metadata = []
        
        # Use ThreadPoolExecutor (I/O bound)
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            # Submit all tasks
            futures = {
                executor.submit(
                    collect_metadata, 
                    row['youtube_video_id'],
                    row['track_id']
                ): idx
                for idx, row in batch_videos.iterrows()
            }
            
            # Collect results with progress bar
            with tqdm(total=len(batch_videos), desc=f"Batch {batch_start//batch_size + 1}") as pbar:
                for future in as_completed(futures):
                    try:
                        metadata = future.result()
                        batch_metadata.append(metadata)
                    except Exception as e:
                        idx = futures[future]
                        row = batch_videos.loc[idx]
                        batch_metadata.append({
                            'track_id': row['track_id'],
                            'youtube_video_id': row['youtube_video_id'],
                            'collection_success': False,
                            'error_message': f'Worker error: {str(e)}'
                        })
                    finally:
                        pbar.update(1)
        
        # Add to all metadata
        all_metadata.extend(batch_metadata)
        
        # Save progress
        metadata_df = pd.DataFrame(all_metadata)
        metadata_df.to_csv(OUTPUT_CSV, index=False)
        
        # Show batch stats
        batch_success = sum(m.get('collection_success', False) for m in batch_metadata)
        batch_blocked = sum('blocked' in m.get('error_message', '').lower() for m in batch_metadata)
        
        print(f"\n✓ Batch complete!")
        print(f"  Total progress: {len(all_metadata):,} videos")
        print(f"  This batch - Success: {batch_success}/{len(batch_metadata)}")
        if batch_blocked > 0:
            print(f"  ⚠️ Blocked: {batch_blocked} videos (YouTube bot detection)")
        print(f"  Saved to: {OUTPUT_CSV}")
        
        # Stop if too many blocks
        if batch_blocked > len(batch_metadata) * 0.5:
            print(f"\n⚠️⚠️⚠️ WARNING: Over 50% blocked in this batch!")
            print(f"  YouTube is blocking requests.")
            print(f"  Recommendation: Stop and try later with VPN/hotspot")
            response = input("\nContinue anyway? (yes/no): ")
            if response.lower() != 'yes':
                print("Stopping...")
                break
    
    return pd.DataFrame(all_metadata)

# Run collection
print("Starting metadata collection...\n")
print(f"Processing {len(pending):,} videos with {NUM_WORKERS} workers\n")

metadata_df = process_batch_metadata(
    pending,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE
)

print(f"\n{'='*70}")
print("METADATA COLLECTION COMPLETE!")
print(f"{'='*70}")

Starting metadata collection...

Processing 11,689 videos with 8 workers

Loaded 31,500 existing metadata records


Batch: 1-500 of 11,689
Workers: 8



Batch 1:   0%|          | 0/500 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 32,000 videos
  This batch - Success: 498/500
  Saved to: ..\data\raw\youtube_metadata.csv

Batch: 501-1000 of 11,689
Workers: 8



Batch 2:   0%|          | 0/500 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 32,500 videos
  This batch - Success: 494/500
  ⚠️ Blocked: 1 videos (YouTube bot detection)
  Saved to: ..\data\raw\youtube_metadata.csv

Batch: 1001-1500 of 11,689
Workers: 8



...


✓ Batch complete!
  Total progress: 42,500 videos
  This batch - Success: 498/500
  ⚠️ Blocked: 1 videos (YouTube bot detection)
  Saved to: ..\data\raw\youtube_metadata.csv

Batch: 11001-11500 of 11,689
Workers: 8



Batch 23:   0%|          | 0/500 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 43,000 videos
  This batch - Success: 498/500
  ⚠️ Blocked: 1 videos (YouTube bot detection)
  Saved to: ..\data\raw\youtube_metadata.csv

Batch: 11501-11689 of 11,689
Workers: 8



Batch 24:   0%|          | 0/189 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 43,189 videos
  This batch - Success: 188/189
  ⚠️ Blocked: 1 videos (YouTube bot detection)
  Saved to: ..\data\raw\youtube_metadata.csv

METADATA COLLECTION COMPLETE!


## 8. Analyze Collected Metadata

In [8]:
# Load and analyze metadata
if OUTPUT_CSV.exists():
    metadata_df = pd.read_csv(OUTPUT_CSV)
    
    print("="*70)
    print("METADATA COLLECTION SUMMARY")
    print("="*70)
    
    print(f"\nTotal videos processed: {len(metadata_df):,}")
    
    if 'collection_success' in metadata_df.columns:
        successful = metadata_df['collection_success'].sum()
        failed = (~metadata_df['collection_success']).sum()
        
        print(f"Successful: {successful:,} ({(successful/len(metadata_df)*100):.1f}%)")
        print(f"Failed: {failed:,} ({(failed/len(metadata_df)*100):.1f}%)")
    
    # Engagement metrics summary
    if 'view_count' in metadata_df.columns:
        print(f"\nEngagement Metrics (successful collections):")
        success_df = metadata_df[metadata_df['collection_success'] == True]
        
        print(f"\nView counts:")
        print(f"  Mean: {success_df['view_count'].mean():,.0f}")
        print(f"  Median: {success_df['view_count'].median():,.0f}")
        print(f"  Max: {success_df['view_count'].max():,.0f}")
        
        print(f"\nLike counts:")
        print(f"  Mean: {success_df['like_count'].mean():,.0f}")
        print(f"  Median: {success_df['like_count'].median():,.0f}")
        
        print(f"\nComment counts:")
        print(f"  Mean: {success_df['comment_count'].mean():,.0f}")
        print(f"  Median: {success_df['comment_count'].median():,.0f}")
    
    print(f"\n✓ Metadata saved to: {OUTPUT_CSV}")
else:
    print("No metadata file found yet - run collection first!")

METADATA COLLECTION SUMMARY

Total videos processed: 43,189
Successful: 42,885 (99.3%)
Failed: 304 (0.7%)

Engagement Metrics (successful collections):

View counts:
  Mean: 28,924,989
  Median: 185,579
  Max: 6,965,757,892

Like counts:
  Mean: 211,700
  Median: 2,224

Comment counts:
  Mean: 9,033
  Median: 191

✓ Metadata saved to: ..\data\raw\youtube_metadata.csv


## 9. Merge All Datasets

**Combine: Spotify + YouTube + Audio Features + Metadata**

In [11]:
import pandas as pd
from pathlib import Path

print("="*70)
print("MERGING ALL DATASETS")
print("="*70)

# Load datasets
print("\n1. Loading datasets...")

spotify_df = pd.read_csv("../data/raw/spotify_tracks.csv")
print(f"   Spotify: {len(spotify_df):,} rows")

download_df = pd.read_csv("../data/raw/combined_match_download_results.csv")
print(f"   Download: {len(download_df):,} rows")

features_df = pd.read_csv("../data/raw/audio_features.csv")
print(f"   Features: {len(features_df):,} rows")

metadata_df = pd.read_csv("../data/raw/youtube_metadata.csv")
print(f"   Metadata: {len(metadata_df):,} rows")

# Step 1: Merge Spotify + Download
print("\n2. Merging Spotify + Download results...")
complete_df = spotify_df.merge(
    download_df, 
    on='track_id', 
    how='left',
    suffixes=('', '_download')
)
print(f"   Result: {len(complete_df):,} rows, {len(complete_df.columns)} columns")

# Step 2: Merge with Audio Features
print("\n3. Merging + Audio features...")

# Only keep successful feature extractions
features_success = features_df[features_df['extraction_success'] == True].copy()
print(f"   Using {len(features_success):,} successful feature extractions")

# Drop non-feature columns before merge
feature_cols_to_drop = ['extraction_success', 'error_message']
features_to_merge = features_success.drop(columns=feature_cols_to_drop, errors='ignore')

complete_df = complete_df.merge(
    features_to_merge,
    on='track_id',
    how='left'
)
print(f"   Result: {len(complete_df):,} rows, {len(complete_df.columns)} columns")

# Step 3: Merge with YouTube Metadata
print("\n4. Merging + YouTube metadata...")

# Only keep successful metadata collections
metadata_success = metadata_df[metadata_df['collection_success'] == True].copy()
print(f"   Using {len(metadata_success):,} successful metadata collections")

# Drop non-metadata columns before merge
metadata_cols_to_drop = ['collection_success', 'error_message', 'track_id']
metadata_to_merge = metadata_success.drop(columns=metadata_cols_to_drop, errors='ignore')

complete_df = complete_df.merge(
    metadata_to_merge,
    on='youtube_video_id',
    how='left',
    suffixes=('', '_metadata')
)
print(f"   Result: {len(complete_df):,} rows, {len(complete_df.columns)} columns")

# Save complete dataset
output_path = Path("../data/raw/complete_dataset.csv")
complete_df.to_csv(output_path, index=False)

print("\n" + "="*70)
print("MERGE COMPLETE!")
print("="*70)

print(f"\nDataset saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024*1024):.1f} MB")
print(f"\nTotal rows: {len(complete_df):,}")
print(f"Total columns: {len(complete_df.columns)}")

# Data completeness
print(f"\nData Completeness:")
print(f"  Total Spotify tracks: {len(complete_df):,}")
print(f"  With YouTube match: {complete_df['youtube_video_id'].notna().sum():,} ({complete_df['youtube_video_id'].notna().sum()/len(complete_df)*100:.1f}%)")
print(f"  With audio features: {complete_df['tempo'].notna().sum():,} ({complete_df['tempo'].notna().sum()/len(complete_df)*100:.1f}%)")
print(f"  With YouTube metadata: {complete_df['view_count'].notna().sum():,} ({complete_df['view_count'].notna().sum()/len(complete_df)*100:.1f}%)")

# Complete records (all data available)
complete_records = complete_df[
    complete_df['youtube_video_id'].notna() & 
    complete_df['tempo'].notna() & 
    complete_df['view_count'].notna()
]
print(f"  Complete records (all data): {len(complete_records):,} ({len(complete_records)/len(complete_df)*100:.1f}%)")

# Show column breakdown
print(f"\nColumn Breakdown:")
print(f"  Spotify metadata: ~20 columns")
print(f"  YouTube download info: ~10 columns")
print(f"  Audio features: 89 columns")
print(f"  YouTube engagement: ~15 columns")
print(f"  Total: {len(complete_df.columns)} columns")

# Sample data
print(f"\nSample of merged data (first 3 rows):")
sample_cols = ['track_name', 'artists', 'popularity', 'tempo', 'view_count', 'like_count']
print(complete_df[sample_cols].head(3).to_string(index=False))

# Engagement stats
print(f"\nYouTube Engagement Statistics (songs with metadata):")
with_metadata = complete_df[complete_df['view_count'].notna()]
print(f"  Average views: {with_metadata['view_count'].mean():,.0f}")
print(f"  Median views: {with_metadata['view_count'].median():,.0f}")
print(f"  Max views: {with_metadata['view_count'].max():,.0f}")

print(f"\n✓ Complete dataset ready for modeling!")

MERGING ALL DATASETS

1. Loading datasets...
   Spotify: 114,000 rows
   Download: 47,344 rows
   Features: 44,271 rows
   Metadata: 43,189 rows

2. Merging Spotify + Download results...
   Result: 119,337 rows, 36 columns

3. Merging + Audio features...
   Using 44,271 successful feature extractions
   Result: 119,337 rows, 125 columns

4. Merging + YouTube metadata...
   Using 42,885 successful metadata collections
   Result: 181,409 rows, 139 columns

MERGE COMPLETE!

Dataset saved to: ..\data\raw\complete_dataset.csv
File size: 1051.9 MB

Total rows: 181,409
Total columns: 139

Data Completeness:
  Total Spotify tracks: 181,409
  With YouTube match: 129,770 (71.5%)


KeyError: 'tempo'

In [18]:
import pandas as pd
from pathlib import Path

print("="*70)
print("CREATING IN-PROGRESS DATASET")
print("="*70)

# Load all datasets
print("\n1. Loading datasets...")
spotify_df = pd.read_csv("../data/raw/spotify_tracks.csv")
download_df = pd.read_csv("../data/raw/combined_match_download_results.csv")
features_df = pd.read_csv("../data/raw/audio_features.csv")
metadata_df = pd.read_csv("../data/raw/youtube_metadata.csv")

print(f"   Spotify: {len(spotify_df):,} tracks")
print(f"   Download: {len(download_df):,} results")
print(f"   Features: {len(features_df):,} songs")
print(f"   Metadata: {len(metadata_df):,} songs")

# Merge everything (left join to keep all Spotify tracks)
print("\n2. Merging all data...")

# Start with Spotify
progress_df = spotify_df.copy()

# Add download info
progress_df = progress_df.merge(
    download_df[['track_id', 'youtube_video_id', 'youtube_url', 'youtube_title']], 
    on='track_id', 
    how='left'
)

# Add audio features (only successful ones)
features_success = features_df[features_df['extraction_success'] == True].copy()
features_clean = features_success.drop(columns=['extraction_success', 'error_message'], errors='ignore')
progress_df = progress_df.merge(features_clean, on='track_id', how='left')

# Add YouTube metadata (only successful ones)
metadata_success = metadata_df[metadata_df['collection_success'] == True].copy()
metadata_clean = metadata_success.drop(columns=['collection_success', 'error_message', 'track_id'], errors='ignore')
progress_df = progress_df.merge(metadata_clean, on='youtube_video_id', how='left')

# Rename tempo columns if they exist
if 'tempo_x' in progress_df.columns:
    progress_df = progress_df.rename(columns={'tempo_x': 'tempo_spotify', 'tempo_y': 'tempo_librosa'})

print(f"   Result: {len(progress_df):,} rows, {len(progress_df.columns)} columns")

# Add status flags
print("\n3. Adding progress tracking columns...")
progress_df['has_youtube_match'] = progress_df['youtube_video_id'].notna()
progress_df['has_audio_features'] = progress_df['tempo_librosa'].notna() if 'tempo_librosa' in progress_df.columns else progress_df['tempo'].notna()
progress_df['has_youtube_metadata'] = progress_df['view_count'].notna()
progress_df['is_complete'] = progress_df['has_youtube_match'] & progress_df['has_audio_features'] & progress_df['has_youtube_metadata']

# Save
output_path = Path("../data/raw/in_progress_dataset.csv")
progress_df.to_csv(output_path, index=False)

print("\n" + "="*70)
print("IN-PROGRESS DATASET CREATED!")
print("="*70)

print(f"\nSaved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024*1024):.1f} MB")

# Show progress statistics
print(f"\n" + "="*70)
print("PROGRESS STATISTICS")
print("="*70)

total = len(progress_df)
with_youtube = progress_df['has_youtube_match'].sum()
with_features = progress_df['has_audio_features'].sum()
with_metadata = progress_df['has_youtube_metadata'].sum()
complete = progress_df['is_complete'].sum()

print(f"\nTotal Spotify tracks: {total:,}")
print(f"\nProgress:")
print(f"  ✓ YouTube matched: {with_youtube:,} ({with_youtube/total*100:.1f}%)")
print(f"  ✓ Audio features extracted: {with_features:,} ({with_features/total*100:.1f}%)")
print(f"  ✓ YouTube metadata collected: {with_metadata:,} ({with_metadata/total*100:.1f}%)")
print(f"  ✓ Complete (all data): {complete:,} ({complete/total*100:.1f}%)")

# Show what's still needed
remaining = total - complete
print(f"\n  ⏳ Still need complete data for: {remaining:,} songs ({remaining/total*100:.1f}%)")

# Sample
print(f"\nSample of dataset:")
sample_cols = ['track_name', 'artists', 'has_youtube_match', 'has_audio_features', 'has_youtube_metadata', 'is_complete']
print(progress_df[sample_cols].head(10).to_string(index=False))

print(f"\n✓ You can now safely delete MP3s!")
print(f"✓ All extracted data is saved in CSV files")

CREATING IN-PROGRESS DATASET

1. Loading datasets...
   Spotify: 114,000 tracks
   Download: 47,344 results
   Features: 44,271 songs
   Metadata: 43,189 songs

2. Merging all data...
   Result: 181,409 rows, 127 columns

3. Adding progress tracking columns...

IN-PROGRESS DATASET CREATED!

Saved to: ..\data\raw\in_progress_dataset.csv
File size: 1034.4 MB

PROGRESS STATISTICS

Total Spotify tracks: 181,409

Progress:
  ✓ YouTube matched: 129,770 (71.5%)
  ✓ Audio features extracted: 128,373 (70.8%)
  ✓ YouTube metadata collected: 127,624 (70.4%)
  ✓ Complete (all data): 127,486 (70.3%)

  ⏳ Still need complete data for: 53,923 songs (29.7%)

Sample of dataset:
                track_name                              artists  has_youtube_match  has_audio_features  has_youtube_metadata  is_complete
                    Comedy                          Gen Hoshino               True                True                  True         True
          Ghost - Acoustic                         Ben

In [19]:
import pandas as pd

df = pd.read_csv("../data/raw/in_progress_dataset.csv")

print("Your Progress:")
print(f"  Songs with features: {df['has_audio_features'].sum():,}")
print(f"  Songs with metadata: {df['has_youtube_metadata'].sum():,}")
print(f"  Complete songs: {df['is_complete'].sum():,}")

Your Progress:
  Songs with features: 128,373
  Songs with metadata: 127,624
  Complete songs: 127,486


In [20]:
import pandas as pd

print("="*70)
print("DEBUGGING FILE CONTENTS")
print("="*70)

# Check each file
print("\n1. Spotify tracks:")
spotify = pd.read_csv("../data/raw/spotify_tracks.csv")
print(f"   Rows: {len(spotify):,}")
print(f"   Unique track_ids: {spotify['track_id'].nunique():,}")

print("\n2. Audio features:")
features = pd.read_csv("../data/raw/audio_features.csv")
print(f"   Total rows: {len(features):,}")
print(f"   Successful extractions: {features['extraction_success'].sum():,}")
print(f"   Failed extractions: {(~features['extraction_success']).sum():,}")
print(f"   Unique track_ids: {features['track_id'].nunique():,}")

print("\n3. YouTube metadata:")
metadata = pd.read_csv("../data/raw/youtube_metadata.csv")
print(f"   Total rows: {len(metadata):,}")
print(f"   Successful collections: {metadata['collection_success'].sum():,}")
print(f"   Failed collections: {(~metadata['collection_success']).sum():,}")
print(f"   Unique track_ids: {metadata['track_id'].nunique():,}")

print("\n4. In-progress dataset:")
progress = pd.read_csv("../data/raw/in_progress_dataset.csv")
print(f"   Total rows: {len(progress):,}")
print(f"   Unique track_ids: {progress['track_id'].nunique():,}")

# Check a specific column
if 'tempo_librosa' in progress.columns:
    print(f"\n   Songs with tempo_librosa (not null): {progress['tempo_librosa'].notna().sum():,}")
elif 'tempo' in progress.columns:
    print(f"\n   Songs with tempo (not null): {progress['tempo'].notna().sum():,}")

print(f"   Songs with view_count (not null): {progress['view_count'].notna().sum():,}")

# Check status flags
if 'has_audio_features' in progress.columns:
    print(f"\n   has_audio_features = True: {progress['has_audio_features'].sum():,}")
    print(f"   has_youtube_metadata = True: {progress['has_youtube_metadata'].sum():,}")
    print(f"   is_complete = True: {progress['is_complete'].sum():,}")

DEBUGGING FILE CONTENTS

1. Spotify tracks:
   Rows: 114,000
   Unique track_ids: 89,741

2. Audio features:
   Total rows: 44,271
   Successful extractions: 44,271
   Failed extractions: 0
   Unique track_ids: 44,271

3. YouTube metadata:
   Total rows: 43,189
   Successful collections: 42,885
   Failed collections: 304
   Unique track_ids: 41,441

4. In-progress dataset:
   Total rows: 181,409
   Unique track_ids: 89,741

   Songs with tempo_librosa (not null): 128,373
   Songs with view_count (not null): 127,624

   has_audio_features = True: 128,373
   has_youtube_metadata = True: 127,624
   is_complete = True: 127,486


In [21]:
import pandas as pd

# Check audio_features for duplicates
features = pd.read_csv("../data/raw/audio_features.csv")
print("Audio Features Check:")
print(f"  Total rows: {len(features):,}")
print(f"  Unique track_ids: {features['track_id'].nunique():,}")
print(f"  Duplicate track_ids: {len(features) - features['track_id'].nunique():,}")

if len(features) > features['track_id'].nunique():
    print(f"\n  ⚠️ WARNING: {len(features) - features['track_id'].nunique():,} duplicate rows!")
    
    # Show duplicates
    duplicates = features[features['track_id'].duplicated(keep=False)]
    print(f"\n  Sample duplicates:")
    print(duplicates[['track_id', 'tempo']].head(10))

# Check metadata for duplicates
metadata = pd.read_csv("../data/raw/youtube_metadata.csv")
print("\n\nYouTube Metadata Check:")
print(f"  Total rows: {len(metadata):,}")
print(f"  Unique track_ids: {metadata['track_id'].nunique():,}")
print(f"  Duplicate track_ids: {len(metadata) - metadata['track_id'].nunique():,}")

if len(metadata) > metadata['track_id'].nunique():
    print(f"\n  ⚠️ WARNING: {len(metadata) - metadata['track_id'].nunique():,} duplicate rows!")

Audio Features Check:
  Total rows: 44,271
  Unique track_ids: 44,271
  Duplicate track_ids: 0


YouTube Metadata Check:
  Total rows: 43,189
  Unique track_ids: 41,441
  Duplicate track_ids: 1,748

  ⚠️ WARNING: 1,748 duplicate rows!


In [22]:
import pandas as pd
from pathlib import Path

print("="*70)
print("FIXING ALL DUPLICATES AND RECREATING DATASET")
print("="*70)

# 1. Clean Spotify (remove duplicates)
print("\n1. Cleaning Spotify dataset...")
spotify = pd.read_csv("../data/raw/spotify_tracks.csv")
print(f"   Before: {len(spotify):,} rows, {spotify['track_id'].nunique():,} unique")

spotify_clean = spotify.drop_duplicates(subset=['track_id'], keep='first')
print(f"   After: {len(spotify_clean):,} rows (removed {len(spotify) - len(spotify_clean):,} duplicates)")

# 2. Clean YouTube metadata (remove duplicates, keep only successful)
print("\n2. Cleaning YouTube metadata...")
metadata = pd.read_csv("../data/raw/youtube_metadata.csv")
print(f"   Before: {len(metadata):,} rows")

metadata_success = metadata[metadata['collection_success'] == True].copy()
metadata_clean = metadata_success.drop_duplicates(subset=['track_id'], keep='first')
print(f"   After: {len(metadata_clean):,} rows (removed {len(metadata_success) - len(metadata_clean):,} duplicates)")

# Save cleaned metadata
metadata_clean.to_csv("../data/raw/youtube_metadata.csv", index=False)
print(f"   ✓ Saved cleaned youtube_metadata.csv")

# 3. Load audio features (already clean)
print("\n3. Loading audio features...")
features = pd.read_csv("../data/raw/audio_features.csv")
features_clean = features[features['extraction_success'] == True].copy()
print(f"   Rows: {len(features_clean):,} (all unique)")

# 4. Load download results
print("\n4. Loading download results...")
download = pd.read_csv("../data/raw/combined_match_download_results.csv")
print(f"   Rows: {len(download):,}")

# Check for duplicates in download
download_clean = download.drop_duplicates(subset=['track_id'], keep='first')
if len(download_clean) < len(download):
    print(f"   Removed {len(download) - len(download_clean):,} duplicate track_ids")

# 5. Create in_progress_dataset with clean data
print("\n5. Creating in_progress_dataset (with clean data)...")

# Start with clean Spotify
progress_df = spotify_clean.copy()
print(f"   Starting with {len(progress_df):,} Spotify tracks")

# Merge download info
progress_df = progress_df.merge(
    download_clean[['track_id', 'youtube_video_id', 'youtube_url', 'youtube_title']], 
    on='track_id', 
    how='left'
)
print(f"   After adding download info: {len(progress_df):,} rows")

# Merge audio features
features_to_merge = features_clean.drop(columns=['extraction_success', 'error_message'], errors='ignore')
progress_df = progress_df.merge(features_to_merge, on='track_id', how='left')
print(f"   After adding audio features: {len(progress_df):,} rows")

# Merge YouTube metadata
metadata_to_merge = metadata_clean.drop(columns=['collection_success', 'error_message', 'track_id'], errors='ignore')
progress_df = progress_df.merge(metadata_to_merge, on='youtube_video_id', how='left')
print(f"   After adding YouTube metadata: {len(progress_df):,} rows")

# Rename tempo columns
if 'tempo_x' in progress_df.columns:
    progress_df = progress_df.rename(columns={
        'tempo_x': 'tempo_spotify',
        'tempo_y': 'tempo_librosa'
    })

# Add status flags
tempo_col = 'tempo_librosa' if 'tempo_librosa' in progress_df.columns else 'tempo'
progress_df['has_youtube_match'] = progress_df['youtube_video_id'].notna()
progress_df['has_audio_features'] = progress_df[tempo_col].notna()
progress_df['has_youtube_metadata'] = progress_df['view_count'].notna()
progress_df['is_complete'] = (
    progress_df['has_youtube_match'] & 
    progress_df['has_audio_features'] & 
    progress_df['has_youtube_metadata']
)

# Verify no duplicates
if len(progress_df) != progress_df['track_id'].nunique():
    print(f"\n   ⚠️ WARNING: Still have duplicates! Removing...")
    progress_df = progress_df.drop_duplicates(subset=['track_id'], keep='first')
    print(f"   After deduplication: {len(progress_df):,} rows")

# Save
output_path = Path("../data/raw/in_progress_dataset.csv")
progress_df.to_csv(output_path, index=False)

print(f"\n✓ Saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / (1024*1024):.1f} MB")

# Show correct statistics
print("\n" + "="*70)
print("CORRECT PROGRESS STATISTICS")
print("="*70)

total = len(progress_df)
with_youtube = progress_df['has_youtube_match'].sum()
with_features = progress_df['has_audio_features'].sum()
with_metadata = progress_df['has_youtube_metadata'].sum()
complete = progress_df['is_complete'].sum()

print(f"\nTotal unique Spotify tracks: {total:,}")
print(f"\nProgress:")
print(f"  ✓ YouTube matched: {with_youtube:,} ({with_youtube/total*100:.1f}%)")
print(f"  ✓ Audio features extracted: {with_features:,} ({with_features/total*100:.1f}%)")
print(f"  ✓ YouTube metadata collected: {with_metadata:,} ({with_metadata/total*100:.1f}%)")
print(f"  ✓ Complete (all data): {complete:,} ({complete/total*100:.1f}%)")

remaining = total - complete
print(f"\n  ⏳ Still need data for: {remaining:,} songs ({remaining/total*100:.1f}%)")

# Verify numbers make sense
print("\n" + "="*70)
print("SANITY CHECK")
print("="*70)

if total == spotify_clean['track_id'].nunique():
    print(f"✓ Row count matches unique Spotify tracks")
else:
    print(f"✗ ERROR: Row count doesn't match!")

if with_features <= 44271:
    print(f"✓ Feature count is reasonable (≤44,271)")
else:
    print(f"✗ ERROR: Too many features!")

if with_metadata <= 43189:
    print(f"✓ Metadata count is reasonable (≤43,189)")
else:
    print(f"✗ ERROR: Too many metadata entries!")

print(f"\n✓ Dataset fixed and ready!")

FIXING ALL DUPLICATES AND RECREATING DATASET

1. Cleaning Spotify dataset...
   Before: 114,000 rows, 89,741 unique
   After: 89,741 rows (removed 24,259 duplicates)

2. Cleaning YouTube metadata...
   Before: 43,189 rows
   After: 41,150 rows (removed 1,735 duplicates)
   ✓ Saved cleaned youtube_metadata.csv

3. Loading audio features...
   Rows: 44,271 (all unique)

4. Loading download results...
   Rows: 47,344
   Removed 1,947 duplicate track_ids

5. Creating in_progress_dataset (with clean data)...
   Starting with 89,741 Spotify tracks
   After adding download info: 89,741 rows
   After adding audio features: 89,741 rows
   After adding YouTube metadata: 109,435 rows

   ⚠️ WARNING: Still have duplicates! Removing...
   After deduplication: 89,741 rows

✓ Saved to: ..\data\raw\in_progress_dataset.csv
  File size: 303.8 MB

CORRECT PROGRESS STATISTICS

Total unique Spotify tracks: 89,741

Progress:
  ✓ YouTube matched: 45,397 (50.6%)
  ✓ Audio features extracted: 44,271 (49.3%)
  

In [23]:
import pandas as pd

# Load the datasets
progress = pd.read_csv("../data/raw/in_progress_dataset.csv")
metadata = pd.read_csv("../data/raw/youtube_metadata.csv")

print("Quick verification:")
print(f"Metadata file has: {len(metadata):,} rows")
print(f"Progress shows metadata for: {progress['view_count'].notna().sum():,} songs")

# Check if there are multiple youtube_video_ids per track_id
download = pd.read_csv("../data/raw/combined_match_download_results.csv")
download_clean = download.drop_duplicates(subset=['track_id'], keep='first')

multi_videos = download.groupby('track_id')['youtube_video_id'].nunique()
tracks_with_multiple = (multi_videos > 1).sum()

print(f"\nTracks with multiple YouTube videos: {tracks_with_multiple}")

if tracks_with_multiple > 0:
    print("  → This is why metadata count is higher!")
    print("  → One track can match multiple YouTube videos")

C:\Users\Ammar\AppData\Local\Temp\ipykernel_19988\3781875684.py:4: DtypeWarning: Columns (21,22,23,117,118,119,122,123,124,125) have mixed types. Specify dtype option on import or set low_memory=False.
  progress = pd.read_csv("../data/raw/in_progress_dataset.csv")


Quick verification:
Metadata file has: 41,150 rows
Progress shows metadata for: 43,638 songs

Tracks with multiple YouTube videos: 108
  → This is why metadata count is higher!
  → One track can match multiple YouTube videos


In [25]:
import pandas as pd
from pathlib import Path

print("="*70)
print("PRE-DELETION VERIFICATION")
print("="*70)

# 1. Check all essential files exist
print("\n1. Checking essential files exist...")

essential_files = {
    'spotify_tracks.csv': '../data/raw/spotify_tracks.csv',
    'combined_match_download_results.csv': '../data/raw/combined_match_download_results.csv',
    'audio_features.csv': '../data/raw/audio_features.csv',
    'youtube_metadata.csv': '../data/raw/youtube_metadata.csv',
    'in_progress_dataset.csv': '../data/raw/in_progress_dataset.csv'
}

all_exist = True
for name, filepath in essential_files.items():
    path = Path(filepath)
    if path.exists():
        size = path.stat().st_size / (1024*1024)
        print(f"  ✓ {name}: {size:.1f} MB")
    else:
        print(f"  ✗ MISSING: {name}")
        all_exist = False

if not all_exist:
    print("\n⚠️ STOP! Some files are missing. Don't delete MP3s yet!")
    exit()

# 2. Verify data integrity
print("\n2. Verifying data integrity...")

# Load datasets
features = pd.read_csv('../data/raw/audio_features.csv')
metadata = pd.read_csv('../data/raw/youtube_metadata.csv')
download = pd.read_csv('../data/raw/combined_match_download_results.csv')

# Check audio features
print(f"\n  Audio Features:")
print(f"    Total rows: {len(features):,}")
print(f"    Successful extractions: {features['extraction_success'].sum():,}")
print(f"    Has 'tempo' column: {'tempo' in features.columns}")
print(f"    Has 'rms_mean' column: {'rms_mean' in features.columns}")
print(f"    Has 'mfcc_1_mean' column: {'mfcc_1_mean' in features.columns}")

# Count actual feature columns
feature_cols = [c for c in features.columns 
                if c not in ['track_id', 'extraction_success', 'error_message']]
print(f"    Total feature columns: {len(feature_cols)}")

if len(feature_cols) < 85:
    print(f"    ⚠️ WARNING: Expected ~89 features, found {len(feature_cols)}")

# Check metadata
print(f"\n  YouTube Metadata:")
print(f"    Total rows: {len(metadata):,}")
print(f"    Successful collections: {metadata['collection_success'].sum():,}")
print(f"    Has 'view_count': {'view_count' in metadata.columns}")
print(f"    Has 'like_count': {'like_count' in metadata.columns}")

# 3. Cross-reference with MP3s
print("\n3. Cross-referencing with MP3 files...")

audio_dir = Path("../data/raw/youtube_audio")
if audio_dir.exists():
    mp3_files = list(audio_dir.glob("*.mp3"))
    mp3_track_ids = {f.stem for f in mp3_files}
    
    print(f"  MP3 files on disk: {len(mp3_files):,}")
    
    # Check if all MP3s have features

PRE-DELETION VERIFICATION

1. Checking essential files exist...
  ✓ spotify_tracks.csv: 19.2 MB
  ✓ combined_match_download_results.csv: 12.8 MB
  ✓ audio_features.csv: 72.8 MB
  ✓ youtube_metadata.csv: 193.9 MB
  ✓ in_progress_dataset.csv: 303.8 MB

2. Verifying data integrity...

  Audio Features:
    Total rows: 44,271
    Successful extractions: 44,271
    Has 'tempo' column: True
    Has 'rms_mean' column: True
    Has 'mfcc_1_mean' column: True
    Total feature columns: 89

  YouTube Metadata:
    Total rows: 41,150
    Successful collections: 41,150
    Has 'view_count': True
    Has 'like_count': True

3. Cross-referencing with MP3 files...
  MP3 files on disk: 44,271


In [26]:
import pandas as pd
import json
from datetime import datetime

# Create batch record
batch_record = {
    "batch_1_your_work": {
        "date_completed": datetime.now().strftime("%Y-%m-%d"),
        "song_range": "0-51,579",
        "downloads": 45790,
        "features_extracted": 44271,
        "metadata_collected": 41150,
        "status": "COMPLETE - MP3s can be deleted"
    },
    "batch_2_for_groupmate": {
        "song_range": "57,001-83,999",
        "total_songs": 26999,
        "status": "TODO - starting after batch 1 MP3 deletion"
    }
}

# Save record
with open('../data/raw/batch_progress.json', 'w') as f:
    json.dump(batch_record, f, indent=2)

print("✓ Batch record saved")
print(json.dumps(batch_record, indent=2))

✓ Batch record saved
{
  "batch_1_your_work": {
    "date_completed": "2026-04-14",
    "song_range": "0-51,579",
    "downloads": 45790,
    "features_extracted": 44271,
    "metadata_collected": 41150,
    "status": "COMPLETE - MP3s can be deleted"
  },
  "batch_2_for_groupmate": {
    "song_range": "57,001-83,999",
    "total_songs": 26999,
    "status": "TODO - starting after batch 1 MP3 deletion"
  }
}


In [27]:
# This part should show:
mp3_track_ids = {f.stem for f in mp3_files}
features_track_ids = set(features[features['extraction_success'] == True]['track_id'])

mp3s_with_features = mp3_track_ids & features_track_ids
mp3s_without_features = mp3_track_ids - features_track_ids

print(f"  MP3s with extracted features: {len(mp3s_with_features):,}")
print(f"  MP3s without features: {len(mp3s_without_features):,}")

# Should show:
# MP3s with features: 44,271
# MP3s without features: 0

  MP3s with extracted features: 44,271
  MP3s without features: 0


In [28]:
import shutil
from pathlib import Path

audio_dir = Path("../data/raw/youtube_audio")

if audio_dir.exists():
    mp3_files = list(audio_dir.glob("*.mp3"))
    file_count = len(mp3_files)
    total_size_gb = sum(f.stat().st_size for f in mp3_files) / (1024**3)
    
    print("="*70)
    print("DELETING MP3 FILES")
    print("="*70)
    
    print(f"\nAbout to delete:")
    print(f"  Files: {file_count:,} MP3s")
    print(f"  Size: {total_size_gb:.2f} GB")
    
    print(f"\n✓ All features extracted and saved")
    print(f"✓ All metadata collected and saved")
    print(f"✓ Safe to delete")
    
    confirm = input("\nType 'DELETE' to confirm: ")
    
    if confirm == "DELETE":
        print(f"\nDeleting...")
        shutil.rmtree(audio_dir)
        print(f"\n✓✓✓ DELETED!")
        print(f"✓ Freed {total_size_gb:.2f} GB of space")
        print(f"✓ Ready for Batch 2 downloads (songs 57,001-83,999)")
    else:
        print("\nCancelled - MP3s kept")
else:
    print("MP3 folder not found or already deleted")

DELETING MP3 FILES

About to delete:
  Files: 44,271 MP3s
  Size: 153.35 GB

✓ All features extracted and saved
✓ All metadata collected and saved
✓ Safe to delete

Deleting...

✓✓✓ DELETED!
✓ Freed 153.35 GB of space
✓ Ready for Batch 2 downloads (songs 57,001-83,999)


## Next Steps

**After metadata collection:**

1. ✅ Complete dataset created with all features
2. ✅ Ready for virality prediction modeling
3. ⏳ Trade data with groupmate for full 114k dataset
4. ⏳ Build prediction models

**Files created:**
- `youtube_metadata.csv` - Engagement metrics for all songs
- `complete_dataset.csv` - Merged Spotify + YouTube + Audio features